In [34]:
# ============================================================
# Gradient Boosting - Manual tuning approach
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test-unlabelled.csv")

target_col = "class"

X = df_train.drop(columns=[target_col])
y = df_train[target_col]

X_test_final = df_test.copy()

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print(y.value_counts(normalize=True))


Train shape: (1000, 610)
Test shape: (2000, 610)
class
A    0.347
B    0.328
C    0.325
Name: proportion, dtype: float64


In [35]:

# ------------------------------------------------------------
# 2. Preprocessing
# ------------------------------------------------------------

numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [36]:

# ------------------------------------------------------------
# 3. Helper function
# ------------------------------------------------------------

def evaluate_gb(params, label):
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state=42, **params))
    ])

    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1
    )

    return {
        "experiment": label,
        **params,
        "train_acc_mean": scores["train_score"].mean(),
        "train_acc_std": scores["train_score"].std(),
        "cv_acc_mean": scores["test_score"].mean(),
        "cv_acc_std": scores["test_score"].std()
    }

results = []


results = []

def try_gb(params, label=None):
    if label is None:
        label = str(params)

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state=42, **params))
    ])

    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1
    )

    row = {
        "experiment": label,
        **params,
        "train_acc": scores["train_score"].mean(),
        "cv_acc": scores["test_score"].mean(),
        "cv_std": scores["test_score"].std()
    }

    results.append(row)

    print("\n" + "="*60)
    print(label)
    print("Params:", params)
    print(f"Train accuracy: {row['train_acc']:.4f}")
    print(f"5-fold CV accuracy: {row['cv_acc']:.4f} ± {row['cv_std']:.4f}")

    if row["train_acc"] - row["cv_acc"] > 0.05:
        print("Possible overfitting: train accuracy much higher than CV.")
    elif row["cv_acc"] < 0.70:
        print("Possible underfitting or weak configuration.")
    else:
        print("Looks reasonably balanced.")

    return row


Los parámetros del modelo de gradient boosting que nos serviran para la configuracion de este y la mejora son los siguientes:
- n_estimators:
- learning_rate:
- max_depth:
- subsample:
- min_samples_leaf:


La idea es encontrar una combinación de estos que nos permita disminuir la varianza y aumentar la precisión pero tratando de evitar el sobreajuste, a mayores hay parámtros altamente dependientes los unos de los otros como el numero de estimadores y el learning rate.

Antes de comenzar con el análisis probamos con un modelo base que contiene valores estandar de estos parametros para poder tener una idea principal de nuestro punto de partida.

In [10]:

# ------------------------------------------------------------
# 4. Baseline
# ------------------------------------------------------------

baseline_params = {
    "n_estimators": 100,
    "learning_rate": 0.1,
    "max_depth": 3,
    "subsample": 1.0,
    "min_samples_leaf": 1
}

results.append(evaluate_gb(baseline_params, "baseline"))
try_gb(baseline_params, "baseline")



baseline
Params: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.8020 ± 0.0229
Possible overfitting: train accuracy much higher than CV.


{'experiment': 'baseline',
 'n_estimators': 100,
 'learning_rate': 0.1,
 'max_depth': 3,
 'subsample': 1.0,
 'min_samples_leaf': 1,
 'train_acc': 1.0,
 'cv_acc': 0.8019999999999999,
 'cv_std': 0.022934689882359402}

Primeramente podemos ver de este primer modelo que obtenemos un train accuracy de 1.000 y un accuracy resultante del cross-validation del 0.802.

Por tanto podemos sacar unas primeras conclusiones que sugieren que los parametros actuales aportan al modelo una gran capacidad de aprendizaje, no obstante, hay una diferencia significativa entre el resultado del train y el del cv que sugieren que se este dando un potencial caso de sobreajuste.

A mayores, mientras que el accuracy del CV no es malo, es mejorable.

El parametro n_estimators rige directamente el tamaño de nuestro modelo pues indica el número de árboles a entrenar en el proceso del gradiente boosting. Actualmente se tienen 100 pero procederemos a probar coon un rango de valores [50, 100, 200, 400].

A mayor número de árboles mayor complejidad del modelo, esto implica que se puede lograr bajar el sesgo, sin embargo esto puede resultar en el sobreajuste, algo que ya sospechamos puede ocurrir con los 100 arboles. 

In [11]:

# ------------------------------------------------------------
# 5. Study n_estimators
# ------------------------------------------------------------

for n in [50, 100, 200, 400]:
    params = baseline_params.copy()
    params["n_estimators"] = n
    results.append(evaluate_gb(params, f"n_estimators={n}"))
    try_gb(params, f"n_estimators={n}")



n_estimators=50
Params: {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.9995
5-fold CV accuracy: 0.8060 ± 0.0271
Possible overfitting: train accuracy much higher than CV.

n_estimators=100
Params: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.8020 ± 0.0229
Possible overfitting: train accuracy much higher than CV.

n_estimators=200
Params: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.7990 ± 0.0159
Possible overfitting: train accuracy much higher than CV.

n_estimators=400
Params: {'n_estimators': 400, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.8070 ± 0.0223
Possible overfitting: train accuracy much higher than CV.


Vemos que en general parece que los diferentes valores del paramtero no produce ninguna mejora ni empeoramiento claro, mientras que vemos que con 50 ya no se alacanza el ajuste perfecto al entrenamiento.

El cv accuracy mayor se obtiene con 400 sin embargo esta cantidad de arboles puede ser demasiado alta y provocar un sobreajuste.

Se estudiarán otros parámetros para probar sus efectos sobre los valores 50, 100, 200 para poder compararlos.

Otro parámetro de gran importancia es la tasa de aprendizaje, los valores altos de esta suelen implicar una mejor generalizacion (bajan la varianza) lo que resolvería el problema del sobreajuste. Dado que estamos partiendo de un valor de 0.1 se probará con un rango que incluya valores más bajos que este. 

In [15]:

# ------------------------------------------------------------
# 6. Study learning_rate
# ------------------------------------------------------------

for lr in [0.1, 0.05, 0.025]:
    for n in [50, 100, 200]:
        params = baseline_params.copy()
        params["learning_rate"] = lr
        params["n_estimators"] = n
        results.append(evaluate_gb(params, f"learning_rate={lr}"))
        try_gb(params, f"learning_rate={lr}")




learning_rate=0.1
Params: {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.9995
5-fold CV accuracy: 0.8060 ± 0.0271
Possible overfitting: train accuracy much higher than CV.

learning_rate=0.1
Params: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.8020 ± 0.0229
Possible overfitting: train accuracy much higher than CV.

learning_rate=0.1
Params: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.7990 ± 0.0159
Possible overfitting: train accuracy much higher than CV.

learning_rate=0.05
Params: {'n_estimators': 50, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.9543
5-fold CV accuracy: 0.7970 ± 0.0204
Possible overfitting: train accuracy much higher than CV.

learning_rate=0.05


En cuanto a learning rate vemos que los mejores resultados se alcanzan los valores 0.05 y 0.025. Tomaremos 0.05 como punto intermedio ya que si es demasiado pequeño el computo se puede hacer demasiado costoso y el aprendizaje lento.

En cuanto al número de estimadores, se obtienen resultados muy mixtos lo que no nos da una clara comprensión de cual funciona mejor, en general esto se debe a que INCLUIR EXPLICACION

Entonces, a medida que disminuimos la tasa de aprendizaje conviene reducir el numero de estimadores. Podemos ver reflejado esto mismo en los resultados anteriores, vemos que las mejores combinaciones son 0.1-50, 0.05-100 y 0.025-200. Entonces usaremos estas combinaciones para el resto del desarrollo. 

In [17]:
# ------------------------------------------------------------
# 7. Study max_depth
# ------------------------------------------------------------

for depth in [1, 2, 3, 4, 5]:
    for n, lr in [(50, 0.1), (100, 0.05), (200, 0.025)]:
        params = baseline_params.copy()
        params["max_depth"] = depth
        params["n_estimators"] = n
        params["learning_rate"] = lr
        results.append(evaluate_gb(params, f"max_depth={depth}"))
        try_gb(params, f"max_depth={depth}")



max_depth=1
Params: {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 1, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.8490
5-fold CV accuracy: 0.7940 ± 0.0185
Possible overfitting: train accuracy much higher than CV.

max_depth=1
Params: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 1, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.8477
5-fold CV accuracy: 0.7900 ± 0.0190
Possible overfitting: train accuracy much higher than CV.

max_depth=1
Params: {'n_estimators': 200, 'learning_rate': 0.025, 'max_depth': 1, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.8482
5-fold CV accuracy: 0.7880 ± 0.0220
Possible overfitting: train accuracy much higher than CV.

max_depth=2
Params: {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 2, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.9365
5-fold CV accuracy: 0.8020 ± 0.0136
Possible overfitting: train accuracy much higher than CV.

max_depth=2
Params: {'n_estimators': 100, 

En este caso obtenemos una mejora más clara en valores de max_depth más elevados, más específicamente en los valores 3 y 4 donde parece alcanzarse el equilibrio. A medida que aumentamos la profundidad de los árboles obtenemos un modelo más expresivo, capaz de capturar interacciones más complejas, sin embargo los valores bajo permiten una mejor generalizacion y mayor robusted, por esto los valores 3 y 4 son un buen intermedio. 

Se observa claramente que si mantenemos la profundidad en 1 los resultados son significativamente peores a los obtenidos hasta ahora.

Usaremos estos dos valores para seguir buscando la mejor configuración para el modelo de gradient boosting.

Otro parámetro interesante es subsample, que define el subconjunto de datos que le llega a cada árbol de la secuencia. Hasta ahora el valor por defecto era 1, es decir, que en cada paso se usa el 100% de los datos. Si usamos valores menores a 1 estamos ante un gradient boosting estocástico lo cual añade aleatoriedad y por tanto reduce la varianza y ayuda a generalizar lo que puede servir para el actual sobreajuste.

In [18]:
# ------------------------------------------------------------
# 8. Study subsample
# ------------------------------------------------------------

for subsample in [1.0, 0.9, 0.8, 0.6]:
    for n, lr in [(100, 0.05), (200, 0.025)]:
        for dep in [3, 4]:
            params = baseline_params.copy()
            params["subsample"] = subsample
            params["n_estimators"] = n
            params["learning_rate"] = lr
            params["max_depth"] = dep
            results.append(evaluate_gb(params, f"subsample={subsample}"))
            try_gb(params, f"subsample={subsample}")



subsample=1.0
Params: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.9992
5-fold CV accuracy: 0.8040 ± 0.0220
Possible overfitting: train accuracy much higher than CV.

subsample=1.0
Params: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 4, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.8040 ± 0.0235
Possible overfitting: train accuracy much higher than CV.

subsample=1.0
Params: {'n_estimators': 200, 'learning_rate': 0.025, 'max_depth': 3, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 0.9995
5-fold CV accuracy: 0.8050 ± 0.0226
Possible overfitting: train accuracy much higher than CV.

subsample=1.0
Params: {'n_estimators': 200, 'learning_rate': 0.025, 'max_depth': 4, 'subsample': 1.0, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.8100 ± 0.0197
Possible overfitting: train accuracy much higher than CV.

subsample=0.9
Params: {'n_est

Podemos ver que el accuracy del cross-validation tiende a mejorar a medida que disminuimos el parametro, es decir, el porcentaje de datos sobre los que se entrena cada arbol. Esto se debe probablemente a la reduccion de la varianza que reduce el sobreajuste observado, vemos que obtenemos tambien train accuracies ligeramente peores tambien debido a la varianza, sin embargo dado que buscamos aumentar el accuracy lo máximo posible, esto no implica un peor modelo. 

De forma más concreta, los valores de 0.6 y 0.8 ofrecen los mejores resultados, llegando al CV accuracy más alto hasta ahora con 0.6 siendo este 0.8170.

El último parámetro a analizar es min_samples_leaf, que hasta ahora ha sido fijado en 1. Este 

In [20]:
# ------------------------------------------------------------
# 9. Study min_samples_leaf
# ------------------------------------------------------------

for leaf in [1, 3, 5, 10, 20]:
    params = baseline_params.copy()
    params["min_samples_leaf"] = leaf
    params["n_estimators"] = 200
    params["learning_rate"] = 0.025
    params["max_depth"] = 3
    params["subsample"] = 0.8
    results.append(evaluate_gb(params, f"min_samples_leaf={leaf}"))
    try_gb(params, f"min_samples_leaf={leaf}")



min_samples_leaf=1
Params: {'n_estimators': 200, 'learning_rate': 0.025, 'max_depth': 3, 'subsample': 0.8, 'min_samples_leaf': 1}
Train accuracy: 1.0000
5-fold CV accuracy: 0.8050 ± 0.0176
Possible overfitting: train accuracy much higher than CV.

min_samples_leaf=3
Params: {'n_estimators': 200, 'learning_rate': 0.025, 'max_depth': 3, 'subsample': 0.8, 'min_samples_leaf': 3}
Train accuracy: 0.9990
5-fold CV accuracy: 0.8150 ± 0.0152
Possible overfitting: train accuracy much higher than CV.

min_samples_leaf=5
Params: {'n_estimators': 200, 'learning_rate': 0.025, 'max_depth': 3, 'subsample': 0.8, 'min_samples_leaf': 5}
Train accuracy: 0.9983
5-fold CV accuracy: 0.8110 ± 0.0215
Possible overfitting: train accuracy much higher than CV.

min_samples_leaf=10
Params: {'n_estimators': 200, 'learning_rate': 0.025, 'max_depth': 3, 'subsample': 0.8, 'min_samples_leaf': 10}
Train accuracy: 0.9940
5-fold CV accuracy: 0.8150 ± 0.0197
Possible overfitting: train accuracy much higher than CV.

min_s

Vemos que en general, el número de muestras en las hojas de los arboles no tiene un efecto significativo, siendo los valores 5, 10 y 20 bastante similares en cuanto a rendimiento. 

Finalmente realizamos un gridsearch sobre los rangos de valores que han dado los mejores resultados y añadiendo ciertos valores que nos permiten seguir buscando mejores resultados en la dirección que nos han sugerido los análisis anteriores.

In [38]:
# ------------------------------------------------------------
# 11. Final small GridSearchCV
# ------------------------------------------------------------

final_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

param_grid = {
    "model__n_estimators": [100, 200, 300, 400],
    "model__learning_rate": [0.05, 0.025],
    "model__max_depth": [3, 4],
    "model__subsample": [0.5, 0.6, 0.7, 0.8],
    "model__min_samples_leaf": [5, 10, 20, 40]
}

grid = GridSearchCV(
    final_pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    verbose=2
)

grid.fit(X, y)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)
print("Best train accuracy:", grid.cv_results_["mean_train_score"][grid.best_index_])

Fitting 5 folds for each of 256 candidates, totalling 1280 fits
[CV] END model__learning_rate=0.05, model__max_depth=3, model__min_samples_leaf=5, model__n_estimators=100, model__subsample=0.5; total time=  23.9s
[CV] END model__learning_rate=0.05, model__max_depth=3, model__min_samples_leaf=5, model__n_estimators=100, model__subsample=0.5; total time=  24.5s
[CV] END model__learning_rate=0.05, model__max_depth=3, model__min_samples_leaf=5, model__n_estimators=100, model__subsample=0.5; total time=  24.5s
[CV] END model__learning_rate=0.05, model__max_depth=3, model__min_samples_leaf=5, model__n_estimators=100, model__subsample=0.5; total time=  24.5s
[CV] END model__learning_rate=0.05, model__max_depth=3, model__min_samples_leaf=5, model__n_estimators=100, model__subsample=0.5; total time=  25.2s
[CV] END model__learning_rate=0.05, model__max_depth=3, model__min_samples_leaf=5, model__n_estimators=100, model__subsample=0.6; total time=  29.6s
[CV] END model__learning_rate=0.05, model_

/opt/anaconda3/lib/python3.12/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best params: {'model__learning_rate': 0.025, 'model__max_depth': 4, 'model__min_samples_leaf': 10, 'model__n_estimators': 400, 'model__subsample': 0.7}
Best CV accuracy: 0.825
Best train accuracy: 1.0


In [39]:
results_df = pd.DataFrame(grid.cv_results_)

# Train accuracy de la mejor configuración en CV (lo que tienes ahora)
best_cv_idx = grid.best_index_
print(f"Mejor CV:    {results_df.loc[best_cv_idx, 'mean_test_score']:.4f}")
print(f"Su train:    {results_df.loc[best_cv_idx, 'mean_train_score']:.4f}")

# Máximo train accuracy global (configuración distinta)
best_train_idx = results_df['mean_train_score'].idxmax()
print(f"\nMáximo train: {results_df.loc[best_train_idx, 'mean_train_score']:.4f}")
print(f"Su CV:         {results_df.loc[best_train_idx, 'mean_test_score']:.4f}")

Mejor CV:    0.8250
Su train:    1.0000

Máximo train: 1.0000
Su CV:         0.8220


In [ ]:

# ------------------------------------------------------------
# 10. Results table
# ------------------------------------------------------------

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="cv_acc_mean", ascending=False)

display(results_df)

results_df.to_csv("gradient_boosting_manual_experiments.csv", index=False)

In [ ]:
# ------------------------------------------------------------
# 12. Final predictions
# ------------------------------------------------------------

best_model = grid.best_estimator_
best_model.fit(X, y)

preds = best_model.predict(X_test_final)

submission = df_test.copy()
submission[target_col] = preds

submission.to_csv("test_gradient_boosting_predictions.csv", index=False)